In [2]:
#!/usr/bin/env python3
"""
Analyzes all simulation_progress_group_XXX.log files and provides a 
comprehensive summary of successes, running tasks, and a detailed list of failures.
"""

import os
import glob
import re
from datetime import datetime

def parse_log_file(log_path):
    """Parse a single log file and extract key information."""
    info = {
        'group': None,
        'total_scenarios': 0,
        'successful': 0,
        'failed': 0,
        'failed_list': [],
        'current_scenario': None,
        'last_progress': None,
        'last_update_time': None,
        'status': 'Unknown',
        'last_line': None
    }
    
    match = re.search(r'group_(\d+)', log_path)
    if match:
        info['group'] = f"group_{match.group(1)}"
    
    if not os.path.exists(log_path):
        info['status'] = 'Not Found'
        return info
    
    mod_time = os.path.getmtime(log_path)
    info['last_update_time'] = datetime.fromtimestamp(mod_time)
    
    try:
        with open(log_path, 'r') as f:
            lines = f.readlines()
        
        if not lines:
            info['status'] = 'Empty'
            return info
        
        info['last_line'] = lines[-1].strip()
        
        for i, line in enumerate(lines):
            # Track Scenario Starts
            if 'STARTING:' in line:
                match = re.search(r'STARTING: (group_\d+) - (\S+)', line)
                if match:
                    info['current_scenario'] = match.group(2)
            
            # Track Progress
            if 'time:' in line and 'days' in line:
                match = re.search(r'time: ([\d.]+)s \(([\d.]+) days, ([\d.]+)%\)', line)
                if match:
                    info['last_progress'] = {
                        'percent': float(match.group(3)),
                        'days': float(match.group(2))
                    }
            
            # Track Success
            if 'COMPLETED & MERGED:' in line:
                info['successful'] += 1
                info['current_scenario'] = None

            # Track Failures (Detects ERROR logs, Tracebacks, or OS errors)
            if any(err in line for err in ['ERROR', 'FAILED', 'Traceback', 'OSError', 'UnicodeDecodeError']):
                # Look backwards for the last 'STARTING' line to find which scenario failed
                search_idx = i
                while search_idx >= 0:
                    if 'STARTING:' in lines[search_idx]:
                        m = re.search(r'STARTING: group_\d+ - (\S+)', lines[search_idx])
                        if m:
                            scen = m.group(1).strip()
                            if scen not in info['failed_list']:
                                info['failed_list'].append(scen)
                                info['failed'] = len(info['failed_list'])
                        break
                    search_idx -= 1

            # Parse Final Summary Block if exists
            if 'Successful:' in line:
                match = re.search(r'Successful: (\d+)/(\d+)', line)
                if match:
                    info['successful'] = int(match.group(1))
                    info['total_scenarios'] = int(match.group(2))
            if 'Failed:' in line:
                match = re.search(r'Failed: (\d+)/(\d+)', line)
                if match:
                    info['failed'] = int(match.group(1))

        # Determine overall Status
        content = "".join(lines[-20:])
        if 'ALL CYCLES COMPLETE' in content or 'ALL SCENARIOS PROCESSED' in content:
            info['status'] = 'Completed'
        elif info['current_scenario']:
            if info['last_progress']:
                info['status'] = f"Running ({info['last_progress']['percent']:.1f}%)"
            else:
                info['status'] = 'Running (Init)'
        else:
            info['status'] = 'Idle/Stalled'
        
    except Exception as e:
        info['status'] = f'Parsing Error: {str(e)}'
    
    return info

def format_time_ago(dt):
    if dt is None: return "Unknown"
    diff = datetime.now() - dt
    s = diff.total_seconds()
    if s < 60: return f"{int(s)}s ago"
    if s < 3600: return f"{int(s/60)}m ago"
    if s < 86400: return f"{int(s/3600)}h ago"
    return f"{int(s/86400)}d ago"

def main():
    log_files = sorted(glob.glob('*_simulation_progress_group_*.log'))
    if not log_files:
        print("No log files found in logs/ directory.")
        return

    print("="*80)
    print(f"{'ANUGA CLUSTER MONITOR':^80}")
    print("="*80)

    global_failed_scenarios = []
    all_info = []

    for log_file in log_files:
        info = parse_log_file(log_file)
        all_info.append(info)
        
        print(f"\nGroup: {info['group']:<15} | Status: {info['status']:<18} | Updated: {format_time_ago(info['last_update_time'])}")
        
        if info['total_scenarios'] > 0:
            print(f"  Success: {info['successful']}/{info['total_scenarios']} | Failed: {info['failed']}")
        
        if info['current_scenario']:
            print(f"  > Active: {info['current_scenario']}")
            
        if info['failed_list']:
            print(f"  > ❌ FAILED: {', '.join(info['failed_list'])}")
            for s in info['failed_list']:
                global_failed_scenarios.append(f"{info['group']}:{s}")

    # OVERALL SUMMARY
    print("\n" + "="*80)
    print(f"{'OVERALL SYSTEM SUMMARY':^80}")
    print("="*80)
    
    total_successful = sum(i['successful'] for i in all_info)
    total_failed = len(global_failed_scenarios)
    completed_groups = sum(1 for i in all_info if i['status'] == 'Completed')
    
    print(f"Total Groups Processed: {len(all_info)}")
    print(f"Groups Fully Finished:  {completed_groups}")
    print(f"Total Successes (SWW):  {total_successful}")
    print(f"Total Failures (ERR):   {total_failed}")

    if global_failed_scenarios:
        print("-" * 80)
        print("DETAILED FAILURE LIST:")
        for fail in global_failed_scenarios:
            print(f"  [!] {fail}")
    
    # Check for stalled jobs
    stalled = [i for i in all_info if i['status'] != 'Completed' and 
               (datetime.now() - i['last_update_time']).total_seconds() > 3600]
    
    if stalled:
        print("-" * 80)
        print("⚠️  STALLED JOBS (No activity > 1 hour):")
        for i in stalled:
            print(f"  - {i['group']} (Last update: {format_time_ago(i['last_update_time'])})")
    
    print("="*80)

if __name__ == "__main__":
    main()

                             ANUGA CLUSTER MONITOR                              

Group: group_000       | Status: Idle/Stalled       | Updated: 1h ago
  Success: 40/40 | Failed: 0

Group: group_001       | Status: Idle/Stalled       | Updated: 1h ago
  Success: 40/40 | Failed: 0

Group: group_002       | Status: Idle/Stalled       | Updated: 1h ago
  Success: 40/40 | Failed: 0

Group: group_003       | Status: Idle/Stalled       | Updated: 45m ago
  Success: 40/40 | Failed: 0

Group: group_004       | Status: Idle/Stalled       | Updated: 1h ago
  Success: 40/40 | Failed: 0

Group: group_005       | Status: Idle/Stalled       | Updated: 56m ago
  Success: 40/40 | Failed: 0

Group: group_006       | Status: Idle/Stalled       | Updated: 39m ago
  Success: 40/40 | Failed: 0

Group: group_007       | Status: Idle/Stalled       | Updated: 1h ago
  Success: 40/40 | Failed: 0

Group: group_008       | Status: Idle/Stalled       | Updated: 1h ago
  Success: 40/40 | Failed: 0

Group: group_00